# 🧬 Protein Structure Prediction & Design
## AlphaFold2 · ESMFold · ProteinMPNN · RoseTTAFold — Industry-Standard Tutorial
### Python Ecosystem Tutorial Series — Module 20

**Author:** Himanshu Goel | [hgoelgithub.github.io](https://hgoelgithub.github.io)

---

| | |
|---|---|
| **Domain** | Structural Bioinformatics / Computational Protein Design |
| **Tools** | AlphaFold2, ESMFold, ProteinMPNN, RoseTTAFold, BioPython, py3Dmol |
| **Sequence** | Thioredoxin (P10599) · Design example: de novo miniprotein |
| **Paradigm** | Structure prediction → inverse folding → sequence design → validation |

## What you will learn

1. The AlphaFold2 revolution — architecture and what changed in biology
2. How to run structure prediction (ESMFold API + AlphaFold ColabFold)
3. Parse and analyse PDB structures with BioPython
4. Protein inverse folding with ProteinMPNN — design sequences for a fixed backbone
5. Evaluate designed sequences: pLDDT, pAE, Rosetta energy, in-silico validation
6. The full protein design pipeline from scratch backbone → optimised sequence

```bash
pip install biopython requests py3Dmol matplotlib numpy pandas scipy
pip install biotite    # fast sequence/structure analysis
# ColabFold (local): conda install -c conda-forge colabfold
# ESMFold API: no install needed — uses Meta's public API
```

---
## 1. The Protein Folding Revolution — What Changed and Why It Matters

### The 50-Year Problem
The **protein folding problem** — predicting a protein's 3D structure from its amino acid
sequence — was biology's grand challenge for 50 years. Anfinsen's 1972 Nobel Prize showed
that sequence determines structure, but *how* remained elusive.

### Before AlphaFold (pre-2020)
```
Experimental structure determination:
  X-ray crystallography  → months to years + luck with crystals
  Cryo-EM               → months + expensive equipment
  NMR spectroscopy      → weeks + size limited to ~50 kDa
  
Computational prediction (homology modelling):
  Works only if a similar sequence has a known structure (>30% identity)
  Completely fails for novel folds
  
CASP13 (2018): best model RMSD ~7 Å on hard targets — not useful for biology
```

### After AlphaFold2 (2021)
```
AlphaFold2 at CASP14 (2020): median RMSD ~1 Å — as accurate as experiments
Structure prediction: 2 minutes (ColabFold) vs 2 years (X-ray)
AlphaFold Database (2022): 200M+ predicted structures — entire known proteome

ESMFold (Meta, 2022):   even faster, language-model based, no MSA needed
RoseTTAFold (UW, 2021): open-source, similar accuracy
OmegaFold (2022):       single-sequence, no alignment needed
```

### The Paradigm Shift for Drug Discovery & Toxicology
```
Before:    drug target has no structure → can't do structure-based design
After:     AlphaFold predicts structure in 2 minutes → dock ligands immediately

Before:    understanding why mutation causes disease requires structure → years
After:     predict mutant structure → measure structural deviation → minutes

Before:    engineer enzyme for bioremediation requires known structure
After:     design enzyme sequence for novel fold → ProteinMPNN → test experimentally
```

### The Protein Design Stack (2024)
```
Sequence only          SEQUENCE → STRUCTURE
─────────────          AlphaFold2, ESMFold, RoseTTAFold
                       Input: FASTA sequence
                       Output: PDB coordinates + confidence (pLDDT, pAE)

Structure → Sequence   STRUCTURE → SEQUENCE  (inverse folding)
────────────────────   ProteinMPNN, LigandMPNN, ESM-IF1
                       Input: PDB backbone coordinates
                       Output: amino acid sequences that fold to that shape

Generative design      NOTHING → SEQUENCE + STRUCTURE
─────────────────      RFdiffusion, ProteinGenerator, Chroma
                       Input: constraints (bind this target, fold type)
                       Output: completely novel protein design
```


---
## 2. AlphaFold2 Architecture — How It Works

### The Key Innovation: Evoformer + Structure Module

AlphaFold2 takes two inputs:
1. **Multiple Sequence Alignment (MSA)**: evolutionary sequences — proteins that folded the same way over billions of years must be constrained by physics
2. **Template structures**: known structures of homologous proteins (optional)

```
Input: amino acid sequence (e.g., 100 residues)
         ↓
MSA Search (jackhmmer/HHblits against UniRef90, BFD, MGnify)
         ↓  (N_seq × L × 23 matrix — evolutionary covariation)

┌─────────────────────────────────────────────────────────────────┐
│                     Evoformer (48 blocks)                       │
│                                                                 │
│  MSA representation   ←→   Pair representation                 │
│  (N_seq × L × d_msa)        (L × L × d_pair)                  │
│                                                                 │
│  MSA row attention          Triangle multiplicative update      │
│  MSA column attention       Triangle self-attention             │
│  Transition                 Outer product mean                  │
│                                                                 │
│  Key insight: pair representation encodes co-evolution          │
│  If residue i and j co-evolved → they're physically close       │
└─────────────────────────────────────────────────────────────────┘
         ↓  single sequence representation (L × d_single)

┌─────────────────────────────────────────────────────────────────┐
│                   Structure Module (8 blocks)                   │
│                                                                 │
│  Invariant Point Attention (IPA)                                │
│  Each residue = rigid body (N, Cα, C backbone frame)           │
│  IPA updates frames in a rotation/translation equivariant way  │
│                                                                 │
│  Output: 3D coordinates for all atoms + side-chain torsion     │
└─────────────────────────────────────────────────────────────────┘
         ↓
Output: PDB coordinates + pLDDT (per-residue confidence 0-100)
                         + pAE  (predicted aligned error, L×L matrix)
```

### Confidence Scores — What They Mean

| Score | Range | Meaning | Use |
|-------|-------|---------|-----|
| **pLDDT** | 0–100 | Per-residue local accuracy | >90: very high, 70–90: good, 50–70: low, <50: disordered |
| **pAE** | 0–31 Å | Relative domain placement | Low=confident relative domain positions |
| **ipTM** | 0–1 | Inter-chain interface quality | >0.8: reliable interface, <0.6: treat with caution |
| **ptm** | 0–1 | Global topology confidence | >0.5: correct fold predicted |

### ESMFold — The Faster Alternative
ESMFold (Meta/FAIR, 2022) uses a **protein language model** (ESM-2, 650M parameters)
instead of MSA search. It's 60× faster than AlphaFold2 and doesn't need a sequence database.
Slightly lower accuracy on hard targets but excellent for rapid screening.

```
AlphaFold2: sequence → MSA search (5-30 min) → Evoformer → structure (1-2 min)
ESMFold:    sequence → ESM-2 embeddings (seconds) → folding trunk → structure (seconds)
```


---
## 3. Structure Prediction with ESMFold API

ESMFold is available as a free REST API from Meta. One POST request → PDB structure.
No installation, no GPU, no database download needed.

**Sequence used:** Human Thioredoxin (UniProt P10599) — 105 residues, well-characterised
oxidoreductase involved in redox signalling. Crystal structure known (1ERU).


In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import warnings, io, re, math, json
from collections import defaultdict
warnings.filterwarnings("ignore")

# ── Human Thioredoxin (UniProt P10599) ───────────────────────────────────────
# 105 aa — well-studied, known crystal structure 1ERU
THIOREDOXIN_SEQ = (
    "MVKQIESKTAFQEALDAAGDKLVVVDFSATWCGPCKMIKPFFHSLSEKYSNVIFLEVDVDDCQDVASECEVK"
    "CMPTFQFFKKGQKVGEFSGANKEKLEATINELV"
)

print(f"Sequence: {THIOREDOXIN_SEQ}")
print(f"Length:   {len(THIOREDOXIN_SEQ)} residues")
print(f"UniProt:  P10599 (Human Thioredoxin)")
print(f"Function: Disulfide oxidoreductase, redox signalling, antioxidant defence")
print()

# ── ESMFold API call ──────────────────────────────────────────────────────────
ESMFOLD_URL = "https://api.esmatlas.com/foldSequence/v1/pdb/"

def predict_structure_esmfold(sequence: str, name: str = "protein") -> dict:
    """
    Submit sequence to ESMFold API and return PDB string + metadata.
    
    Args:
        sequence: amino acid sequence (single-letter codes)
        name:     protein name for logging
    
    Returns:
        dict with 'pdb_string', 'success', 'error' keys
    """
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    try:
        response = requests.post(
            ESMFOLD_URL,
            data=sequence,
            headers=headers,
            timeout=120
        )
        if response.status_code == 200:
            pdb_text = response.text
            print(f"✅ ESMFold prediction succeeded for {name}")
            print(f"   PDB size: {len(pdb_text):,} characters")
            return {"pdb_string": pdb_text, "success": True, "error": None}
        else:
            print(f"⚠️  ESMFold API returned {response.status_code} — using simulated data")
            return {"pdb_string": None, "success": False,
                    "error": f"HTTP {response.status_code}"}
    except Exception as e:
        print(f"⚠️  ESMFold API not reachable ({e}) — using simulated structure data")
        return {"pdb_string": None, "success": False, "error": str(e)}

result = predict_structure_esmfold(THIOREDOXIN_SEQ, "Thioredoxin P10599")
print()
print("API Response:", "Success — PDB received" if result["success"] else f"Offline — {result['error']}")


---
## 4. Parsing PDB Structures with BioPython

BioPython's `Bio.PDB` module provides a complete hierarchy for working with structures:

```
Structure
 └── Model (different conformations, e.g. NMR)
      └── Chain (polypeptide chains A, B, C...)
           └── Residue (amino acids, waters, ligands)
                └── Atom (N, CA, C, O, side-chain atoms...)
                     └── coordinates (x, y, z) + B-factor + occupancy
```

This is the standard format for all PDB file analysis in Python.


In [ ]:
try:
    from Bio.PDB import PDBParser, PPBuilder, PDBIO, Select
    from Bio.PDB.vectors import Vector
    from Bio import SeqIO
    from Bio.SeqUtils.ProtParam import ProteinAnalysis
    BIOPYTHON_OK = True
    print(f"BioPython available")
except ImportError:
    BIOPYTHON_OK = False
    print("pip install biopython")

# ── Generate a realistic synthetic PDB for the tutorial ──────────────────────
# (In real use: pdb_text comes from ESMFold API or PDB download)

def generate_thioredoxin_pdb(sequence: str) -> str:
    """
    Generate a realistic PDB file for thioredoxin.
    Coordinates follow an idealised alpha-beta topology (TRX fold).
    Used when ESMFold API is unavailable.
    """
    # TRX fold: beta1-alpha1-beta2-beta3-alpha2-beta4-alpha3
    # Secondary structure assignment (simplified)
    SS_MAP = {
        range(1,7):   "E",  # β1
        range(8,20):  "H",  # α1
        range(21,27): "E",  # β2
        range(30,36): "E",  # β3
        range(37,55): "H",  # α2
        range(56,62): "E",  # β4
        range(67,82): "H",  # α3
        range(90,105):"H",  # α4 C-term
    }
    def get_ss(i):
        for rng, ss in SS_MAP.items():
            if i in rng: return ss
        return "C"

    # pLDDT values (realistic ESMFold output for thioredoxin)
    # High in structured regions, lower in loops
    np.random.seed(42)
    base_plddt = np.array([
        72,75,80,85,88,90,92,91,94,96,97,95,93,94,95,96,94,92,88,85,
        87,89,92,94,93,91,80,75,72,78,83,87,90,92,94,96,97,96,95,94,
        93,94,95,96,95,94,92,90,88,86,85,84,87,89,92,93,92,90,88,85,
        83,82,80,78,80,83,87,90,92,94,95,94,92,90,88,85,83,80,78,76,
        75,77,80,83,87,90,92,94,93,92,90,88,85,83,80,78,76,74,72,70,
        72,74,76,78,80,
    ], dtype=float)
    plddt = np.clip(base_plddt[:len(sequence)] + np.random.randn(len(sequence))*2, 20, 100)

    lines = ["REMARK  ESMFold prediction (simulated) for P10599 Human Thioredoxin",
             "REMARK  pLDDT values embedded in B-factor column"]

    # Generate Cα trace coordinates along idealized path
    coords = []
    t = np.linspace(0, 4*np.pi, len(sequence))
    x_base = 12 * np.cos(t*0.5)
    y_base = 12 * np.sin(t*0.5)
    z_base = 3.8 * t / (4*np.pi) * len(sequence) / 10

    for i in range(len(sequence)):
        ss = get_ss(i+1)
        if ss == "H":   # helix: tighter coil
            x = x_base[i] + 2.3*np.cos(i*100*np.pi/180)
            y = y_base[i] + 2.3*np.sin(i*100*np.pi/180)
            z = z_base[i] + 1.5*i/len(sequence)
        elif ss == "E": # strand: extended
            x = x_base[i] + 0.5*(i%2-0.5)
            y = y_base[i]
            z = z_base[i]
        else:           # coil: irregular
            x = x_base[i] + np.random.uniform(-1.5,1.5)
            y = y_base[i] + np.random.uniform(-1.5,1.5)
            z = z_base[i]
        coords.append((x, y, z))

    atom_num = 1
    for i, (aa, (x,y,z), plddt_i) in enumerate(zip(sequence, coords, plddt)):
        resnum = i + 1
        lines.append(
            f"ATOM  {atom_num:5d}  CA  {_aa1to3(aa)} A{resnum:4d}    "
            f"{x:8.3f}{y:8.3f}{z:8.3f}  1.00{plddt_i:6.2f}           C"
        )
        atom_num += 1
    lines.append("END")
    return "\n".join(lines)

def _aa1to3(aa):
    """Convert 1-letter amino acid code to 3-letter."""
    map3 = {
        'A':'ALA','C':'CYS','D':'ASP','E':'GLU','F':'PHE','G':'GLY','H':'HIS',
        'I':'ILE','K':'LYS','L':'LEU','M':'MET','N':'ASN','P':'PRO','Q':'GLN',
        'R':'ARG','S':'SER','T':'THR','V':'VAL','W':'TRP','Y':'TYR','X':'UNK'
    }
    return map3.get(aa, 'UNK')

# Use API result if available, else generate
if result["success"] and result["pdb_string"]:
    PDB_TEXT = result["pdb_string"]
    DATA_SOURCE = "ESMFold API"
else:
    PDB_TEXT = generate_thioredoxin_pdb(THIOREDOXIN_SEQ)
    DATA_SOURCE = "Simulated (ESMFold-format)"

print(f"Structure source: {DATA_SOURCE}")
print(f"PDB lines: {len(PDB_TEXT.splitlines())}")
print()
print("First 5 ATOM lines:")
for line in PDB_TEXT.splitlines():
    if line.startswith("ATOM"):
        print(f"  {line}")
    if line.startswith("ATOM") and "  5 " in line[:15]:
        break


In [ ]:
# ── Parse structure with BioPython ────────────────────────────────────────────

def parse_pdb_from_string(pdb_text: str, name: str = "structure"):
    """Parse PDB text into BioPython structure object."""
    pdb_io = io.StringIO(pdb_text)
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(name, pdb_io)
    return structure

def extract_ca_coords(structure) -> np.ndarray:
    """Extract Cα coordinates as numpy array (N × 3)."""
    coords = []
    for model in structure:
        for chain in model:
            for residue in chain:
                if "CA" in residue:
                    coords.append(residue["CA"].get_vector().get_array())
    return np.array(coords)

def extract_plddt(structure) -> np.ndarray:
    """Extract pLDDT from B-factor column (ESMFold convention)."""
    plddt = []
    for model in structure:
        for chain in model:
            for residue in chain:
                if "CA" in residue:
                    plddt.append(residue["CA"].get_bfactor())
    return np.array(plddt)

def assign_secondary_structure_simple(coords: np.ndarray) -> list:
    """
    Simplified secondary structure assignment from Cα coordinates.
    Uses Cα-Cα-Cα angles and distances.
    In production: use DSSP via BioPython or mkdssp.
    """
    n = len(coords)
    ss = ["C"] * n
    for i in range(1, n-1):
        v1 = coords[i]   - coords[i-1]
        v2 = coords[i+1] - coords[i]
        dist = np.linalg.norm(v2)
        cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8)
        angle_deg = np.degrees(np.arccos(np.clip(cos_angle, -1, 1)))
        # Helix: ~3.8 Å between Cα, angle ~50°
        # Strand: ~3.8 Å between Cα, angle ~120°
        if dist < 4.1 and angle_deg < 80:
            ss[i] = "H"
        elif dist < 4.2 and angle_deg > 100:
            ss[i] = "E"
    # Smooth (remove single-residue islands)
    for i in range(1, n-1):
        if ss[i-1] == ss[i+1] != ss[i]:
            ss[i] = ss[i-1]
    return ss

if BIOPYTHON_OK:
    struct  = parse_pdb_from_string(PDB_TEXT, "thioredoxin")
    ca_coords = extract_ca_coords(struct)
    plddt_vals = extract_plddt(struct)
    ss_assign  = assign_secondary_structure_simple(ca_coords)
else:
    # Manual parsing fallback
    ca_coords, plddt_vals = [], []
    for line in PDB_TEXT.splitlines():
        if line.startswith("ATOM") and " CA " in line:
            try:
                x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
                b = float(line[60:66])
                ca_coords.append([x, y, z])
                plddt_vals.append(b)
            except: pass
    ca_coords  = np.array(ca_coords)
    plddt_vals = np.array(plddt_vals)
    ss_assign  = assign_secondary_structure_simple(ca_coords)

# ── Structural statistics ──────────────────────────────────────────────────────
ss_counts = {s: ss_assign.count(s) for s in "HEC"}
rg = np.sqrt(np.mean(np.sum((ca_coords - ca_coords.mean(0))**2, axis=1)))
dist_matrix = np.sqrt(((ca_coords[:,None] - ca_coords[None,:])**2).sum(-1))

print("── Structural Statistics ──")
print(f"Residues:          {len(ca_coords)}")
print(f"Mean pLDDT:        {plddt_vals.mean():.1f}")
print(f"pLDDT ≥ 90 ('very high'): {(plddt_vals >= 90).sum()} residues ({(plddt_vals>=90).mean():.0%})")
print(f"pLDDT ≥ 70 ('confident'):  {(plddt_vals >= 70).sum()} residues ({(plddt_vals>=70).mean():.0%})")
print(f"Radius of gyration: {rg:.1f} Å")
print(f"Max Cα-Cα distance: {dist_matrix.max():.1f} Å")
print(f"Secondary structure: {ss_counts['H']} residues H | {ss_counts['E']} residues E | {ss_counts['C']} residues coil")


---
## 5. Protein Sequence Analysis — From Sequence to Function

Before designing new sequences, we characterise the wild-type protein thoroughly.
This establishes baseline properties that designed sequences must match or improve.


In [ ]:
# ── Physicochemical characterisation ──────────────────────────────────────────
AA_PROPERTIES = {
    "hydrophobic": set("AVILMFYWC"),
    "polar":       set("STNQ"),
    "charged_pos": set("RKH"),
    "charged_neg": set("DE"),
    "aromatic":    set("FYW"),
    "tiny":        set("AGST"),
    "aliphatic":   set("ILVA"),
}

AA_HYDROPHOBICITY = {
    "A":1.8,"R":-4.5,"N":-3.5,"D":-3.5,"C":2.5,"Q":-3.5,"E":-3.5,
    "G":-0.4,"H":-3.2,"I":4.5,"L":3.8,"K":-3.9,"M":1.9,"F":2.8,
    "P":-1.6,"S":-0.8,"T":-0.7,"W":-0.9,"Y":-1.3,"V":4.2
}

def analyse_sequence(seq: str) -> dict:
    n = len(seq)
    aa_counts = {aa: seq.count(aa) for aa in "ACDEFGHIKLMNPQRSTVWY"}
    
    # Composition by property
    comp = {k: sum(seq.count(aa) for aa in chars)/n
            for k, chars in AA_PROPERTIES.items()}
    
    # Average hydrophobicity (Kyte-Doolittle scale)
    hydrophob = np.mean([AA_HYDROPHOBICITY.get(aa, 0) for aa in seq])
    
    # Isoelectric point (simplified)
    pos_charge = seq.count("R") + seq.count("K") + seq.count("H")*0.1
    neg_charge = seq.count("D") + seq.count("E")
    
    # Instability index (Guruprasad et al. 1990)
    # DIWV table for dipeptide instability
    DIPEPTIDE_INSTABILITY = {
        "WW":1.0,"WC":1.0,"CF":1.0,"CW":1.0,"WM":1.0,
        "CD":1.0,"EC":1.0,"CE":1.0,"CY":1.0,"WD":1.0,
    }
    instability = 10/n * sum(DIPEPTIDE_INSTABILITY.get(seq[i:i+2], 0)
                              for i in range(n-1))
    
    return {
        "length": n,
        "aa_counts": aa_counts,
        "composition": comp,
        "hydrophobicity": hydrophob,
        "pos_charge": pos_charge,
        "neg_charge": neg_charge,
        "net_charge": pos_charge - neg_charge,
        "instability_index": instability,
        "n_cysteines": seq.count("C"),
        "disulfide_bonds_possible": seq.count("C") // 2,
    }

props = analyse_sequence(THIOREDOXIN_SEQ)

print("── Human Thioredoxin (P10599) Sequence Analysis ──")
print(f"Length:               {props['length']} aa")
print(f"Net charge (pH 7):    {props['net_charge']:+.1f}")
print(f"Hydrophobicity (avg): {props['hydrophobicity']:.2f} (Kyte-Doolittle)")
print(f"Instability index:    {props['instability_index']:.1f} (<40 = stable)")
print(f"Cysteines:            {props['n_cysteines']} ({props['disulfide_bonds_possible']} possible S-S bonds)")
print()
print("Composition by property:")
for k, v in props["composition"].items():
    bar = "█" * int(v*30)
    print(f"  {k:15s}: {v:.1%} {bar}")

# Sliding window hydrophobicity profile (window=9)
seq = THIOREDOXIN_SEQ
W = 9
hydro_profile = []
for i in range(len(seq)-W+1):
    window = seq[i:i+W]
    hydro_profile.append(np.mean([AA_HYDROPHOBICITY.get(aa, 0) for aa in window]))
hydro_x = np.arange(W//2, len(seq)-W//2)

# Conservation simulation (would come from MSA in real analysis)
np.random.seed(99)
conservation = 0.5 + 0.4*np.abs(np.sin(np.linspace(0, 3*np.pi, len(seq))))
# Active site residues are highly conserved
active_site_res = [29, 31]  # Cys32, Cys35 in thioredoxin (0-indexed: 28, 31)
for r in active_site_res:
    conservation[r] = 0.99

print()
print(f"Active site: Cys32-Gly33-Pro34-Cys35 (CGPC motif at positions 31-34)")
print(f"These cysteines form the redox-active disulfide bond")


---
## 6. ProteinMPNN — Inverse Folding (Backbone → Sequence)

**ProteinMPNN** (Dauparas et al., Science 2022) is the industry-standard inverse folding tool.
Given a fixed protein backbone (Cα, N, C, O atoms), it designs amino acid sequences
that are predicted to fold into that exact structure.

### How ProteinMPNN Works
```
Input: backbone coordinates (Cα, N, C, O for each residue)
         ↓
Edge construction: k-nearest neighbours graph (k=48 by default)
Each node = residue frame, each edge = spatial relationship
         ↓
Message Passing Neural Network (3 encoder + 3 decoder layers)
Encoder: aggregates structural context into node embeddings
Decoder: autoregressively samples amino acids (position 1→2→...→N)
         ↓
Output: probability distribution P(aa_i | backbone, aa_1...aa_{i-1})
        Sample multiple sequences at temperature T
        Higher T = more diversity, lower T = more conservative
```

### Key ProteinMPNN Parameters
| Parameter | Default | Effect |
|-----------|---------|--------|
| `temperature` | 0.1 | Sampling diversity. 0.1=conservative, 0.5=diverse |
| `num_seq_per_target` | 8 | How many sequences to generate |
| `backbone_noise` | 0.0 | Add Å noise to coords for robustness |
| `fixed_residues` | [] | Positions to keep as wild-type |
| `tied_positions` | [] | Force identical AAs (for symmetry) |
| `chain_mask` | 1 | Which chains to design |

### Applications
- **Protein stabilisation**: design thermostable variants of enzymes
- **De novo design**: design sequences for completely new backbones from RFdiffusion
- **Antibody optimisation**: improve CDR loop sequences for affinity/stability
- **Enzyme engineering**: change active-site loop sequences while preserving fold


In [ ]:
# ── ProteinMPNN simulation ────────────────────────────────────────────────────
# In production: run ProteinMPNN locally:
#   git clone https://github.com/dauparas/ProteinMPNN
#   python protein_mpnn_run.py --pdb_path input.pdb --out_folder output/
#
# Here we simulate the output probability distributions
# based on the known amino acid preferences at each position in thioredoxin

AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY")

def simulate_proteinmpnn_output(sequence: str, ca_coords: np.ndarray,
                                 plddt: np.ndarray, temperature: float = 0.1,
                                 n_sequences: int = 8, seed: int = 42) -> dict:
    """
    Simulate ProteinMPNN output:
    - Log-probability matrix (L × 20) for each amino acid at each position
    - Sampled sequences at given temperature
    - Per-position sequence recovery score
    """
    np.random.seed(seed)
    n = len(sequence)
    
    # Build realistic log-probability matrix
    # High pLDDT → model is confident → lower entropy (sharper distribution)
    # Lower pLDDT → more uncertain → higher entropy
    logprobs = np.zeros((n, 20))
    
    for i, (aa, pld) in enumerate(zip(sequence, plddt)):
        # Baseline: uniform
        logits = np.random.randn(20) * 0.5
        
        # Prefer wild-type amino acid (stronger preference where pLDDT is high)
        wt_idx = AA_ORDER.index(aa) if aa in AA_ORDER else 0
        wt_boost = (pld / 100) * 3.0
        logits[wt_idx] += wt_boost
        
        # Conservative substitutions (similar biochemical properties)
        SIMILAR = {
            "L":["I","V","M"],"I":["L","V","M"],"V":["I","L","A"],
            "A":["V","G","S"],"G":["A","S"],
            "D":["E","N"],"E":["D","Q"],
            "K":["R"],"R":["K"],
            "F":["Y","W"],"Y":["F","W"],
            "S":["T"],"T":["S","V"],
            "H":["R","K"],"N":["Q","D"],"Q":["N","E"],
            "C":["S","A"],"M":["L","I"],
            "P":["G","A"],"W":["F","Y"],
        }
        for sim_aa in SIMILAR.get(aa, []):
            if sim_aa in AA_ORDER:
                sim_idx = AA_ORDER.index(sim_aa)
                logits[sim_idx] += 0.8
        
        # Special: active site cysteines (Cys32, Cys35) are essentially fixed
        if aa == "C" and i in [31, 34]:
            logits[:] = -10
            logits[AA_ORDER.index("C")] = 5.0
        
        # Convert to log-probabilities
        logprobs[i] = logits - np.log(np.exp(logits).sum())
    
    # Sample sequences at temperature T
    sampled_seqs = []
    for s in range(n_sequences):
        np.random.seed(seed + s)
        seq_sample = ""
        for i in range(n):
            # Temperature scaling
            scaled_logits = logprobs[i] / temperature
            probs = np.exp(scaled_logits - scaled_logits.max())
            probs /= probs.sum()
            aa_idx = np.random.choice(20, p=probs)
            seq_sample += AA_ORDER[aa_idx]
        sampled_seqs.append(seq_sample)
    
    # Sequence recovery: fraction matching wild-type
    recovery = [sum(a==b for a,b in zip(sequence, s))/len(sequence)
                for s in sampled_seqs]
    
    # Per-position score: P(wild-type | backbone)
    wt_recovery_per_pos = np.array([
        np.exp(logprobs[i, AA_ORDER.index(aa)] if aa in AA_ORDER else logprobs[i,0])
        for i, aa in enumerate(sequence)
    ])
    
    return {
        "logprobs":          logprobs,
        "sampled_sequences": sampled_seqs,
        "recovery_scores":   recovery,
        "wt_recovery_per_pos": wt_recovery_per_pos,
        "temperature":       temperature,
        "n_sequences":       n_sequences,
    }

# ── Run ProteinMPNN at three temperatures ─────────────────────────────────────
mpnn_results = {}
for T in [0.1, 0.2, 0.5]:
    mpnn_results[T] = simulate_proteinmpnn_output(
        THIOREDOXIN_SEQ, ca_coords, plddt_vals,
        temperature=T, n_sequences=8
    )

T_show = 0.2
res = mpnn_results[T_show]

print(f"ProteinMPNN results (temperature={T_show})")
print(f"{'#':3s} {'Sequence':25s} {'Recovery':10s} {'Status':15s}")
print("-" * 60)
for i, (seq, rec) in enumerate(zip(res["sampled_sequences"], res["recovery_scores"])):
    status = "⭐ Best" if i==np.argmax(res["recovery_scores"]) else ""
    short  = seq[:25] + "..." if len(seq)>25 else seq
    print(f"{i+1:3d} {short:25s} {rec:.1%}      {status}")

print()
best_idx = np.argmax(res["recovery_scores"])
best_seq = res["sampled_sequences"][best_idx]
print(f"Best designed sequence:")
print(f"  WT:  {THIOREDOXIN_SEQ}")
print(f"  Des: {best_seq}")
print()

# Show mutations
mutations = [(i+1, wt, des) for i,(wt,des) in enumerate(zip(THIOREDOXIN_SEQ,best_seq)) if wt!=des]
print(f"Mutations ({len(mutations)} positions):")
for pos, wt, des in mutations[:10]:
    print(f"  {wt}{pos}{des}")
if len(mutations) > 10:
    print(f"  ... and {len(mutations)-10} more")


---
## 7. Validating Designed Sequences — Self-Consistency pLDDT

The key validation step for designed sequences is the **self-consistency test**:

```
1. Start with backbone B
2. Use ProteinMPNN to design sequence S for backbone B
3. Fold S with AlphaFold2 / ESMFold → get predicted structure S'
4. Compare B vs S': compute TM-score and RMSD
```

If TM-score(B, S') > 0.8 and RMSD < 2 Å → the designed sequence is validated!
It independently folds back to the intended backbone.

### Metrics for Design Quality

| Metric | Good threshold | What it measures |
|--------|---------------|-----------------|
| **scTM** | > 0.8 | Self-consistency TM-score |
| **scRMSD** | < 2.0 Å | Cα RMSD between design and AF2 structure |
| **pLDDT** | > 70 (mean) | AlphaFold confidence in designed sequence |
| **Sequence recovery** | 30-50% | How many WT positions were kept |
| **MPNN score** | < −1.5 nats | Log-likelihood of sequence given backbone |

> **The 30-50% recovery sweet spot:** 100% recovery = wild-type. 0% = likely unfolded.
> The best designs are in the 30-50% range — similar to natural homologs.


In [ ]:
# ── Self-consistency validation ────────────────────────────────────────────────

def tm_score_from_coords(coords1: np.ndarray, coords2: np.ndarray) -> float:
    """
    Approximate TM-score between two Cα coordinate sets.
    TM-score(coords1 vs coords2) — topology similarity measure.
    TM-score = 1: identical structures
    TM-score < 0.2: random similarity
    TM-score > 0.5: same fold
    TM-score > 0.8: very similar structures
    """
    n = min(len(coords1), len(coords2))
    c1 = coords1[:n]
    c2 = coords2[:n]
    
    # Translate to centroid
    c1 = c1 - c1.mean(0)
    c2 = c2 - c2.mean(0)
    
    # Kabsch alignment (optimal rotation)
    H = c1.T @ c2
    U, S, Vt = np.linalg.svd(H)
    # Correct for reflection
    d = np.linalg.det(Vt.T @ U.T)
    D = np.eye(3); D[2,2] = d
    R = Vt.T @ D @ U.T
    c1_rot = c1 @ R.T
    
    # TM-score
    d0 = 1.24 * (n - 15)**(1/3) - 1.8  # normalisation factor
    d0 = max(d0, 0.5)
    di = np.sqrt(((c1_rot - c2)**2).sum(axis=1))
    tm = np.mean(1 / (1 + (di/d0)**2))
    return float(tm)

def simulate_folded_coords(sequence: str, seed: int = 0) -> np.ndarray:
    """
    Simulate ESMFold-predicted coordinates for a designed sequence.
    In reality: POST sequence to ESMFold API → parse PDB → extract Cα.
    """
    np.random.seed(seed)
    n = len(sequence)
    # Designed sequence folds to similar structure as WT with some deviation
    coords = ca_coords[:n].copy()
    # Add noise proportional to sequence divergence
    wt_div = sum(a!=b for a,b in zip(THIOREDOXIN_SEQ[:n],sequence[:n]))/n
    noise_scale = 1.0 + wt_div * 3.0  # more mutations → more structural deviation
    coords += np.random.randn(n, 3) * noise_scale * 0.8
    return coords

# ── Validate all 8 designed sequences ─────────────────────────────────────────
print("Self-consistency validation of ProteinMPNN designed sequences")
print(f"{'#':3s} {'Recovery':10s} {'scTM':8s} {'scRMSD':8s} {'pLDDT':8s} {'Status':15s}")
print("─" * 58)

validation_records = []
for i, (des_seq, rec) in enumerate(zip(res["sampled_sequences"],
                                        res["recovery_scores"])):
    # Simulate folding the designed sequence
    des_coords = simulate_folded_coords(des_seq, seed=i)
    n = min(len(ca_coords), len(des_coords))
    
    # Compute self-consistency TM-score
    sc_tm = tm_score_from_coords(ca_coords[:n], des_coords[:n])
    
    # Compute RMSD after alignment
    c1 = ca_coords[:n] - ca_coords[:n].mean(0)
    c2 = des_coords[:n] - des_coords[:n].mean(0)
    H = c1.T @ c2
    U, S, Vt = np.linalg.svd(H)
    d = np.linalg.det(Vt.T @ U.T)
    D = np.eye(3); D[2,2] = d
    R = Vt.T @ D @ U.T
    sc_rmsd = float(np.sqrt(np.mean(((c1 @ R.T) - c2)**2)))
    
    # Simulated pLDDT for designed sequence
    sim_plddt = np.clip(plddt_vals[:n].mean() - (1-rec)*15 + np.random.randn()*3, 50, 98)
    
    # Validation status
    if sc_tm > 0.8 and sc_rmsd < 2.0 and sim_plddt > 70:
        status = "✅ Validated"
    elif sc_tm > 0.6 and sim_plddt > 65:
        status = "⚠️  Marginal"
    else:
        status = "❌ Failed"
    
    validation_records.append({
        "index": i+1, "sequence": des_seq, "recovery": rec,
        "scTM": sc_tm, "scRMSD": sc_rmsd, "pLDDT": sim_plddt,
        "status": status.replace("✅ ","").replace("⚠️  ","").replace("❌ ",""),
        "passed": "Validated" in status
    })
    print(f"{i+1:3d} {rec:.1%}      {sc_tm:.3f}    {sc_rmsd:.2f} Å   {sim_plddt:.0f}       {status}")

df_val = pd.DataFrame(validation_records)
print()
print(f"Validated designs: {df_val['passed'].sum()}/{len(df_val)}")
print(f"Mean scTM:    {df_val['scTM'].mean():.3f}")
print(f"Mean scRMSD:  {df_val['scRMSD'].mean():.2f} Å")


---
## 8. Contact Map & Predicted Aligned Error (pAE) Analysis

The **contact map** shows which residues are spatially close in 3D space.
The **pAE (Predicted Aligned Error)** matrix from AlphaFold shows how
confident the model is about the relative position of every pair of residues.

These are crucial for:
- Identifying hydrogen bond networks and hydrophobic cores
- Assessing domain boundary confidence
- Detecting potentially disordered regions
- Validating the predicted fold class


In [ ]:
# ── Contact map & structural analysis ────────────────────────────────────────

def compute_contact_map(ca_coords: np.ndarray, threshold: float = 8.0) -> np.ndarray:
    """Compute Cα-Cα contact map. Threshold: 8.0 Å is standard."""
    dist_matrix = np.sqrt(((ca_coords[:,None] - ca_coords[None,:])**2).sum(-1))
    return (dist_matrix < threshold).astype(float), dist_matrix

def simulate_pae_matrix(n: int, plddt: np.ndarray) -> np.ndarray:
    """
    Simulate pAE (Predicted Aligned Error) matrix.
    Real pAE comes from AlphaFold2 or ESMFold JSON output.
    pAE[i,j] = expected position error (Å) at residue i if structure aligned at j.
    Low pAE = confident relative placement.
    """
    np.random.seed(42)
    # Base: intra-domain contacts have low pAE, distant/disordered regions higher
    pae = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            # Distance-based pAE baseline
            dist = abs(i-j)
            base = 2.0 + 0.05 * dist
            # Low pLDDT regions have higher pAE
            conf_factor = (200 - plddt[i] - plddt[min(j, n-1)]) / 100
            pae[i,j] = np.clip(base * (1 + conf_factor * 0.5) + 
                                np.random.exponential(0.5), 0, 25)
    # Make symmetric
    pae = (pae + pae.T) / 2
    return pae

# Compute maps
contact_map, dist_matrix = compute_contact_map(ca_coords, threshold=8.0)
pae_matrix = simulate_pae_matrix(len(ca_coords), plddt_vals)

# Analyse secondary structure contacts
n = len(ca_coords)
print("── Contact Map Analysis ──")
print(f"Threshold: 8 Å (Cα-Cα)")
print(f"Total contacts: {int(contact_map.sum()//2)}")
print()

# Short, medium, long range contacts
short_range  = sum(contact_map[i,j] for i in range(n) for j in range(i+1,min(i+6,n)))
medium_range = sum(contact_map[i,j] for i in range(n) for j in range(min(i+6,n),min(i+24,n)))
long_range   = sum(contact_map[i,j] for i in range(n) for j in range(min(i+24,n),n))

print(f"Short range  (|i-j| ≤ 5):  {int(short_range):4d} ({short_range/(short_range+medium_range+long_range):.0%})")
print(f"Medium range (6-23):        {int(medium_range):4d} ({medium_range/(short_range+medium_range+long_range):.0%})")
print(f"Long range   (≥24):         {int(long_range):4d} ({long_range/(short_range+medium_range+long_range):.0%})")
print()
print(f"Mean pAE: {pae_matrix.mean():.1f} Å")
print(f"pAE < 5 Å (high confidence): {(pae_matrix < 5).mean():.0%} of pairs")


In [ ]:
# ── Comprehensive visualisation dashboard ────────────────────────────────────
fig = plt.figure(figsize=(20, 18))
gs  = gridspec.GridSpec(4, 4, hspace=0.52, wspace=0.42)

DARK = "#0D1117"; MED = "#161B22"; BORDER = "#30363D"
C_HIGH = "#2EA44F"; C_MED = "#F0883E"; C_LOW = "#F85149"
C_BLUE = "#58A6FF"; C_PURPLE = "#8957E5"

fig.patch.set_facecolor(DARK)

# ── Panel 1: Pipeline overview ─────────────────────────────────────────────────
ax0 = fig.add_subplot(gs[0, :])
ax0.set_facecolor(DARK); ax0.set_xlim(0,20); ax0.set_ylim(0,3); ax0.axis("off")

steps = [
    (1.5,  "SEQUENCE\nP10599", "#3FB950",  "1"),
    (4.5,  "ESMFold\nPrediction",  "#58A6FF", "2"),
    (7.5,  "BioPython\nAnalysis",  "#8957E5", "3"),
    (10.5, "ProteinMPNN\nDesign",  "#F0883E", "4"),
    (13.5, "Self-Consistency\nValidation", "#3FB950", "5"),
    (16.5, "Ranked\nDesigns", "#D29922", "6"),
    (19.0, "→ Lab", "#F85149", ""),
]
for x, label, col, num in steps:
    if num:
        circ = plt.Circle((x, 1.5), 0.9, color=col, alpha=0.85, zorder=3)
        ax0.add_patch(circ)
        ax0.text(x, 1.8, num, ha="center", va="center",
                  color="white", fontsize=10, fontweight="bold", zorder=4)
        ax0.text(x, 0.7, label, ha="center", va="center",
                  color="#8B949E", fontsize=7.5, zorder=4)
    else:
        ax0.text(x, 1.5, label, ha="center", va="center",
                  color=col, fontsize=12, fontweight="bold")
    if steps.index((x,label,col,num)) < len(steps)-2:
        ax0.annotate("", xy=(x+1.35,1.5), xytext=(x+0.9,1.5),
                      arrowprops=dict(arrowstyle="->",color="#8B949E",lw=2))

ax0.text(10, 2.8, "AlphaFold2 / ProteinMPNN Protein Design Pipeline",
          ha="center", fontsize=12, fontweight="bold", color="white")

# ── Panel 2: 3D structure (Cα trace, coloured by pLDDT) ──────────────────────
ax1 = fig.add_subplot(gs[1, 0], projection="3d")
ax1.set_facecolor(DARK)

# Colour by pLDDT (blue=low, red=high like AlphaFold default)
plddt_norm = (plddt_vals - 50) / 50
cmap_plddt = plt.get_cmap("RdYlBu")
colours = [cmap_plddt(p) for p in plddt_norm]

# Draw Cα trace
for i in range(len(ca_coords)-1):
    x = [ca_coords[i,0], ca_coords[i+1,0]]
    y = [ca_coords[i,1], ca_coords[i+1,1]]
    z = [ca_coords[i,2], ca_coords[i+1,2]]
    ax1.plot(x, y, z, c=colours[i], lw=2, alpha=0.9)
ax1.scatter(*ca_coords.T, c=colours, s=12, alpha=0.8, depthshade=True)

# Highlight active site cysteines
for idx in [31, 34]:
    if idx < len(ca_coords):
        ax1.scatter(*ca_coords[idx], c="yellow", s=120, zorder=5,
                     edgecolors="white", lw=1)
ax1.set_title("Cα Trace (pLDDT colour)
Yellow = active-site Cys", 
               color="white", fontsize=8, fontweight="bold")
for spine in ["x","y","z"]:
    getattr(ax1, f"set_{spine}label")(spine.upper(), color="#8B949E", fontsize=7)
ax1.tick_params(colors="#8B949E", labelsize=6)

# ── Panel 3: pLDDT profile ────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 1])
ax2.set_facecolor(MED)
plddt_colors = ["#2196F3" if p >= 90 else "#03DAC6" if p >= 70 else
                 "#FFC107" if p >= 50 else "#F44336" for p in plddt_vals]
ax2.bar(range(len(plddt_vals)), plddt_vals, color=plddt_colors, alpha=0.9, width=1.0)
ax2.axhline(90, c=C_HIGH,  ls="--", lw=1.5, alpha=0.8, label="Very high (90)")
ax2.axhline(70, c=C_MED,   ls="--", lw=1.5, alpha=0.8, label="Confident (70)")
ax2.axhline(50, c=C_LOW,   ls="--", lw=1.5, alpha=0.8, label="Low (50)")
ax2.set_xlabel("Residue", color="#8B949E", fontsize=8)
ax2.set_ylabel("pLDDT", color="#8B949E", fontsize=8)
ax2.set_title(f"ESMFold pLDDT Profile
Mean={plddt_vals.mean():.1f}",
               color="white", fontsize=8, fontweight="bold")
ax2.legend(fontsize=6, loc="lower right")
ax2.set_facecolor(MED); ax2.tick_params(colors="#8B949E", labelsize=7)
ax2.spines[:].set_color(BORDER)

# ── Panel 4: Secondary structure ──────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 2])
ax3.set_facecolor(MED)
ss_colors = {"H":"#E74C3C","E":"#3498DB","C":"#95A5A6"}
for i, ss in enumerate(ss_assign):
    ax3.bar(i, 1, color=ss_colors[ss], alpha=0.9, width=1.0)
ax3.set_xlim(0, len(ss_assign))
ax3.set_ylim(0, 1.5)
ax3.set_xlabel("Residue", color="#8B949E", fontsize=8)
ax3.set_title(f"Secondary Structure (simple assign)
H={ss_counts['H']} E={ss_counts['E']} C={ss_counts['C']}",
               color="white", fontsize=8, fontweight="bold")
for ss_label, col in ss_colors.items():
    ax3.bar(0, 0, color=col, label={"H":"Helix","E":"Strand","C":"Coil"}[ss_label])
ax3.legend(fontsize=7); ax3.tick_params(colors="#8B949E", labelsize=7)
ax3.spines[:].set_color(BORDER)

# ── Panel 5: Amino acid composition ───────────────────────────────────────────
ax_comp = fig.add_subplot(gs[1, 3])
ax_comp.set_facecolor(MED)
aa_freq = np.array([THIOREDOXIN_SEQ.count(aa)/len(THIOREDOXIN_SEQ) for aa in AA_ORDER])
cols_aa = [("#E74C3C" if aa in "DE" else "#3498DB" if aa in "RKH" else
             "#F1C40F" if aa in "AVILMFW" else "#27AE60") for aa in AA_ORDER]
ax_comp.bar(AA_ORDER, aa_freq, color=cols_aa, alpha=0.85, edgecolor=DARK)
ax_comp.set_xlabel("Amino Acid", color="#8B949E", fontsize=7)
ax_comp.set_ylabel("Frequency", color="#8B949E", fontsize=8)
ax_comp.set_title("AA Composition
(Red=−, Blue=+, Yellow=hydrophobic)",
                   color="white", fontsize=8, fontweight="bold")
ax_comp.tick_params(colors="#8B949E", labelsize=6)
ax_comp.spines[:].set_color(BORDER)

# ── Panel 6: Contact map ──────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[2, 0])
ax4.set_facecolor(MED)
im4 = ax4.imshow(dist_matrix[:60,:60], cmap="viridis_r", vmin=0, vmax=25)
ax4.contour(contact_map[:60,:60], levels=[0.5], colors="white", linewidths=0.5, alpha=0.5)
plt.colorbar(im4, ax=ax4, shrink=0.8, label="Å")
ax4.set_title("Cα Distance Matrix (8Å contacts)
(First 60 residues)",
               color="white", fontsize=8, fontweight="bold")
ax4.set_xlabel("Residue j", color="#8B949E", fontsize=7)
ax4.set_ylabel("Residue i", color="#8B949E", fontsize=7)
ax4.tick_params(colors="#8B949E", labelsize=6)

# ── Panel 7: pAE matrix ───────────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 1])
ax5.set_facecolor(MED)
im5 = ax5.imshow(pae_matrix, cmap="Greens_r", vmin=0, vmax=15)
plt.colorbar(im5, ax=ax5, shrink=0.8, label="Å")
ax5.set_title(f"Predicted Aligned Error (pAE)
Mean={pae_matrix.mean():.1f} Å",
               color="white", fontsize=8, fontweight="bold")
ax5.set_xlabel("Residue j", color="#8B949E", fontsize=7)
ax5.set_ylabel("Residue i", color="#8B949E", fontsize=7)
ax5.tick_params(colors="#8B949E", labelsize=6)

# ── Panel 8: ProteinMPNN recovery by temperature ──────────────────────────────
ax6 = fig.add_subplot(gs[2, 2])
ax6.set_facecolor(MED)
temps = sorted(mpnn_results.keys())
all_recoveries = [mpnn_results[T]["recovery_scores"] for T in temps]
violin_data = ax6.violinplot(all_recoveries, positions=temps,
                               widths=0.08, showmeans=True)
for pc in violin_data["bodies"]:
    pc.set_facecolor(C_BLUE); pc.set_alpha(0.7)
violin_data["cmeans"].set_color("white")
ax6.set_xlabel("Temperature", color="#8B949E", fontsize=8)
ax6.set_ylabel("Sequence Recovery", color="#8B949E", fontsize=8)
ax6.set_title("ProteinMPNN Recovery
vs Temperature",
               color="white", fontsize=8, fontweight="bold")
ax6.tick_params(colors="#8B949E", labelsize=7); ax6.spines[:].set_color(BORDER)

# ── Panel 9: Validation scatter (scTM vs scRMSD) ─────────────────────────────
ax7 = fig.add_subplot(gs[2, 3])
ax7.set_facecolor(MED)
sc_tm_vals   = df_val["scTM"].values
sc_rmsd_vals = df_val["scRMSD"].values
rec_vals     = df_val["recovery"].values
sc = ax7.scatter(sc_rmsd_vals, sc_tm_vals, c=rec_vals, cmap="RdYlGn",
                   s=120, alpha=0.9, edgecolors="white", lw=1, vmin=0.3, vmax=0.7)
plt.colorbar(sc, ax=ax7, shrink=0.8, label="Recovery")
ax7.axhline(0.8, c=C_HIGH, ls="--", lw=1.5, alpha=0.8, label="scTM 0.8")
ax7.axvline(2.0, c=C_LOW,  ls="--", lw=1.5, alpha=0.8, label="RMSD 2 Å")
ax7.fill_between([0, 2.0], [0.8, 0.8], [1.1, 1.1],
                  color=C_HIGH, alpha=0.08, label="Validated zone")
ax7.set_xlabel("scRMSD (Å)", color="#8B949E", fontsize=8)
ax7.set_ylabel("scTM-score", color="#8B949E", fontsize=8)
ax7.set_title("Self-Consistency Validation
scTM vs scRMSD",
               color="white", fontsize=8, fontweight="bold")
ax7.legend(fontsize=6); ax7.tick_params(colors="#8B949E", labelsize=7)
ax7.spines[:].set_color(BORDER)

# ── Panel 10: Per-position wt recovery heatmap ───────────────────────────────
ax8 = fig.add_subplot(gs[3, :3])
ax8.set_facecolor(MED)
# Recovery per position (average across sequences)
per_pos_recovery = np.array([
    np.mean([int(s[i]==THIOREDOXIN_SEQ[i]) for s in res["sampled_sequences"]])
    for i in range(len(THIOREDOXIN_SEQ))
])
# Color strip by secondary structure
for i, ss in enumerate(ss_assign):
    ax8.bar(i, 0.15, bottom=0, color=ss_colors[ss], alpha=0.5, width=1)
ax8.bar(range(len(per_pos_recovery)), per_pos_recovery, bottom=0.15,
         color=[plt.cm.RdYlGn(v) for v in per_pos_recovery], alpha=0.85, width=1)
# Active site markers
for idx, label in [(31,"C32"),(34,"C35")]:
    if idx < len(per_pos_recovery):
        ax8.axvline(idx, c="yellow", ls="--", lw=1.5, alpha=0.8)
        ax8.text(idx, 1.08, label, ha="center", color="yellow", fontsize=8, fontweight="bold")
ax8.set_xlabel("Residue Position", color="#8B949E", fontsize=8)
ax8.set_ylabel("Recovery Rate", color="#8B949E", fontsize=8)
ax8.set_title("Per-position Wild-type Recovery  (top: recovery, bottom: SS assignment  H=red E=blue C=grey)",
               color="white", fontsize=8, fontweight="bold")
ax8.set_xlim(0, len(per_pos_recovery))
ax8.set_ylim(0, 1.2)
ax8.tick_params(colors="#8B949E", labelsize=7); ax8.spines[:].set_color(BORDER)

# ── Panel 11: Summary table ───────────────────────────────────────────────────
ax9 = fig.add_subplot(gs[3, 3])
ax9.set_facecolor(DARK); ax9.axis("off")
summary_data = [
    ("Sequence length",      f"{len(THIOREDOXIN_SEQ)} aa"),
    ("Mean pLDDT",            f"{plddt_vals.mean():.1f}"),
    ("pLDDT ≥ 90",            f"{(plddt_vals>=90).mean():.0%}"),
    ("Total contacts (8Å)",   f"{int(contact_map.sum()//2)}"),
    ("Mean pAE",              f"{pae_matrix.mean():.1f} Å"),
    ("Designs generated",     "8 × 3 temps = 24"),
    ("Validated (T=0.2)",     f"{df_val['passed'].sum()}/8"),
    ("Mean scTM",             f"{df_val['scTM'].mean():.3f}"),
    ("Mean scRMSD",           f"{df_val['scRMSD'].mean():.2f} Å"),
    ("Active site conserved", "C32, C35 (100%)"),
]
ax9.text(0.5, 0.97, "Pipeline Summary", ha="center", va="top",
          fontsize=10, fontweight="bold", color="white", transform=ax9.transAxes)
for i, (key, val) in enumerate(summary_data):
    y = 0.87 - i * 0.09
    ax9.text(0.05, y, key, color="#8B949E", fontsize=7.5, transform=ax9.transAxes)
    ax9.text(0.95, y, val, color="white", fontsize=7.5, ha="right",
              fontweight="bold", transform=ax9.transAxes)
    ax9.axhline(y - 0.01, color=BORDER, lw=0.5, xmin=0.04, xmax=0.96,
                 transform=ax9.transAxes)

plt.suptitle(
    "AlphaFold2 · ESMFold · ProteinMPNN — Protein Structure Prediction & Design Pipeline
"
    "Human Thioredoxin P10599 · 105 residues · TRX fold",
    fontsize=13, fontweight="bold", color="white", y=0.995
)
plt.savefig("alphafold_proteinmpnn_dashboard.png", dpi=120, bbox_inches="tight",
             facecolor=DARK)
plt.show()
print("Dashboard saved: alphafold_proteinmpnn_dashboard.png")


---
## 9. The Full Tool Landscape — When to Use What

### Structure Prediction Tools

| Tool | Speed | Accuracy | Key advantage | When to use |
|------|-------|----------|---------------|-------------|
| **AlphaFold2** | 5-30 min (w/ MSA) | State-of-art | Most accurate on hard targets | Production, publication-quality |
| **ColabFold** | 2-5 min | ≈AF2 | Faster MSA via MMseqs2 | Standard choice for most use cases |
| **ESMFold** | 10-30 sec | Very good | No MSA needed, fastest | Rapid screening of 1000s of sequences |
| **RoseTTAFold** | 10-20 min | Near AF2 | Open-source, multimer, diffusion backbone | Protein-protein complexes |
| **OmegaFold** | ~1 min | Good | Single-sequence, no alignment | Metagenomic sequences |

### Inverse Folding (Sequence Design)
| Tool | Speed | Key advantage | When to use |
|------|-------|---------------|-------------|
| **ProteinMPNN** | Seconds | Best-in-class, most citations | Standard for all backbone→sequence |
| **LigandMPNN** | Seconds | Aware of ligand/cofactor context | Enzyme active-site design |
| **ESM-IF1** | Seconds | Language model embeddings | Alternative when ProteinMPNN unavailable |
| **Rosetta FastRelax** | Minutes | Physics-based energy | Validation + refinement |

### Generative Design (De Novo)
| Tool | Key innovation | When to use |
|------|---------------|-------------|
| **RFdiffusion** | Diffusion on protein backbone frames | De novo binder design, symmetric assemblies |
| **ProteinGenerator** | Sequence + structure jointly | Unconstrained de novo design |
| **Chroma** | Diffusion + programmable constraints | Controlled generation |
| **FrameDiff** | SE(3) diffusion | Research, novel folds |

### Complete Protein Design Workflow
```
1. Define target fold or function
        ↓
2. Generate backbone:
   - Known fold: take from PDB
   - New fold: RFdiffusion / FrameDiff
        ↓
3. Design sequences:
   ProteinMPNN (or LigandMPNN if there's a cofactor)
   Sample 100-1000 sequences
        ↓
4. Self-consistency filter:
   Fold all sequences with ESMFold (fast)
   Keep those with scTM > 0.8
        ↓
5. Rank by:
   pLDDT, MPNN score, Rosetta energy, solubility prediction
        ↓
6. Select 10-50 for experimental testing
   Order synthetic genes (Twist, IDT)
   Express in E. coli, test folding + function
```


---
## 10. Applications in Computational Toxicology & Drug Discovery

### AlphaFold Enabling New Toxicology Science

**1. Protein target identification for toxic compounds**
```python
# Workflow: small molecule toxin → identify binding protein
# Before AF2: needed crystal structure of target
# After AF2:  predict structure of any protein → dock toxin → rank binding sites
from rdkit import Chem
from rdkit.Chem import AllChem
import py3Dmol

# Predict CYP2E1 structure (key metabolic enzyme for chemical toxicity)
cyp2e1_seq = "MSALGVTVALLVACSLTFCHLHQYQ..."  # UniProt P05181
# cyp2e1_pdb = esmfold_predict(cyp2e1_seq)
# dock_ligand(cyp2e1_pdb, acetaminophen_mol)
```

**2. Variant effect prediction for toxic SNPs**
```python
# Mutant structure prediction
wt_seq  = "MVKQIESKTAFQEALD..."            # wild-type
mut_seq = wt_seq.replace("CGPC", "SGPC")  # Cys32Ser variant — destroys active site
# wt_struct  = esmfold_predict(wt_seq)
# mut_struct = esmfold_predict(mut_seq)
# Compare: RMSD, buried surface area, cavity volumes
```

**3. PFAS protein binding — a real toxicology application**
PFOA (perfluorooctanoic acid) disrupts thyroid hormone transport.
With AlphaFold + molecular docking:
- Predict transthyretin (TTR) structure
- Dock PFOA computationally
- Rank PFAS analogues by predicted binding affinity
- Correlate with epidemiological endpoint data

**4. De novo enzyme design for bioremediation**
Design enzymes that degrade PFAS, pesticides, or plastic monomers:
```
1. Identify known dehalogenase or esterase with target activity
2. Extract backbone of active site region
3. RFdiffusion: generate new scaffolds around the active site
4. ProteinMPNN: design sequences
5. Screen in silico for thermostability, solubility
6. Express top candidates → test degradation activity in vitro
```


In [ ]:
# ── Variant effect prediction — toxicology example ──────────────────────────
print("Variant Effect Prediction for Thioredoxin Active Site")
print("="*60)
print()
print("Wild-type sequence around active site:")
print(f"  Position 28-38: {THIOREDOXIN_SEQ[28:38]}  (CGPC motif = Cys32-Gly33-Pro34-Cys35)")
print()

# Define variants of interest (toxicologically relevant mutations)
VARIANTS = [
    ("C32S",   30, "C", "S", "Serine — no longer redox-active, observed in some cancers"),
    ("C35S",   33, "C", "S", "Second active-site cysteine ablation"),
    ("C32A",   30, "C", "A", "Alanine substitution — tests steric effects"),
    ("P34A",   32, "P", "A", "Proline → Alanine — alters backbone geometry"),
    ("G33D",   31, "G", "D", "Glycine → Aspartate — adds charge in active site"),
    ("K36R",   34, "K", "R", "Conservative — preserves charge"),
    ("W31A",   29, "W", "A", "Tryptophan → Alanine — removes aromatic packing"),
]

def predict_variant_effect(wt_seq: str, pos: int, wt_aa: str, mut_aa: str,
                             wt_coords: np.ndarray, wt_plddt: np.ndarray) -> dict:
    """
    Predict structural effect of a point mutation.
    In production: fold mutant with ESMFold → compare to WT structure.
    Here: estimate from position properties.
    """
    np.random.seed(pos*7 + ord(mut_aa))
    
    # Verify wild-type amino acid
    actual_wt = wt_seq[pos] if pos < len(wt_seq) else "?"
    if actual_wt != wt_aa:
        return {"error": f"WT mismatch: expected {wt_aa}, found {actual_wt}"}
    
    # Is this position in the active site?
    is_active_site = pos in [30, 31, 32, 33, 34]
    
    # Local structural context from pLDDT
    local_plddt = wt_plddt[pos] if pos < len(wt_plddt) else 70
    
    # Estimate structural impact (simplified SIFT/PolyPhen-style)
    AA_CHARGE = {"D":-1,"E":-1,"R":1,"K":1,"H":0.5}
    wt_charge  = AA_CHARGE.get(wt_aa, 0)
    mut_charge = AA_CHARGE.get(mut_aa, 0)
    
    AA_SIZE = {"G":60,"A":89,"V":117,"L":131,"I":131,"M":149,
               "P":115,"F":165,"W":204,"Y":181,"S":105,"T":119,
               "C":121,"N":132,"Q":146,"D":133,"E":147,"K":146,"R":174,"H":155}
    size_diff = abs(AA_SIZE.get(mut_aa, 120) - AA_SIZE.get(wt_aa, 120))
    
    # Predict effects
    delta_stability = (
        (-2.5 if is_active_site else -0.5) +          # active site penalty
        (-0.05 * size_diff) +                          # size clash
        (-1.0 * abs(mut_charge - wt_charge)) +         # charge change
        (-1.5 if wt_aa=="C" else 0) +                  # removing Cys = loss of S-S
        (0.5 if wt_aa in "ILVM" and mut_aa in "ILVM" else 0) + # conservative aliphatic
        np.random.normal(0, 0.3)
    )
    
    delta_plddt = np.clip(delta_stability * 2 + np.random.normal(0, 1.5), -20, 5)
    sc_tm_mut   = np.clip(0.92 + delta_stability*0.04 + np.random.normal(0,0.02), 0.5, 0.99)
    sc_rmsd_mut = np.clip(-delta_stability*0.3 + np.random.exponential(0.3), 0.1, 5.0)
    
    effect = "DELETERIOUS" if delta_stability < -2 else              "POSSIBLY DAMAGING" if delta_stability < -1 else "BENIGN"
    
    return {
        "variant":         f"{wt_aa}{pos+1}{mut_aa}",
        "is_active_site":  is_active_site,
        "delta_stability": round(delta_stability, 2),
        "delta_plddt":     round(delta_plddt, 1),
        "sc_tm":           round(sc_tm_mut, 3),
        "sc_rmsd":         round(sc_rmsd_mut, 2),
        "effect":          effect,
        "local_plddt":     round(local_plddt, 1),
    }

print(f"{'Variant':8s} {'AS':4s} {'ΔStab':7s} {'ΔpLDDT':8s} {'scTM':7s} {'scRMSD':8s} {'Effect':20s}")
print("─" * 75)
variant_results = []
for var_name, pos, wt, mut, desc in VARIANTS:
    vr = predict_variant_effect(THIOREDOXIN_SEQ, pos, wt, mut, ca_coords, plddt_vals)
    if "error" not in vr:
        variant_results.append(vr)
        as_flag = "⚡" if vr["is_active_site"] else "  "
        eff_col = ("❌" if "DELETERIOUS" in vr["effect"] else
                   "⚠️ " if "POSSIBLY" in vr["effect"] else "✅")
        print(f"{vr['variant']:8s} {as_flag:4s} {vr['delta_stability']:+7.2f} "
              f"{vr['delta_plddt']:+8.1f} {vr['sc_tm']:7.3f} {vr['sc_rmsd']:6.2f} Å  "
              f"{eff_col} {vr['effect']}")
        print(f"         {desc[:65]}")
        print()

df_var = pd.DataFrame(variant_results)
print(f"\nSummary: {(df_var['effect']=='DELETERIOUS').sum()} deleterious, "
      f"{(df_var['effect']=='POSSIBLY DAMAGING').sum()} possibly damaging, "
      f"{(df_var['effect']=='BENIGN').sum()} benign")


---
## 🧠 Deep Dive — AlphaFold2 & ProteinMPNN: The Science

### Why AlphaFold2 Works — The Key Insight

The Evoformer encodes a single critical piece of information: **co-evolution**.
If two positions in a protein are in physical contact, mutations at one position
are often compensated by mutations at the other over evolutionary time.
This co-evolutionary signal, extracted from thousands of related sequences,
directly encodes the 3D contact map.

The **Invariant Point Attention (IPA)** in the structure module is the second
key innovation. Rather than learning absolute coordinates (which don't generalise
between different protein orientations), IPA learns relationships between rigid
body frames in SE(3) — the group of 3D rotations and translations. This makes
the network equivariant to the input orientation.

### Why pLDDT Is Not Just an Accuracy Score

pLDDT measures **predicted local accuracy** — how confident the model is in its
own prediction. Critically, **low pLDDT often means the region is genuinely
disordered**, not just a failure of the model. Intrinsically disordered regions
(IDRs) have been validated by NMR and other methods to actually lack defined
structure in solution. AlphaFold correctly identifies these regions as uncertain.

For drug discovery: flexible loops often become ordered upon ligand binding.
If AlphaFold shows a low-pLDDT binding site loop, molecular dynamics simulations
are needed to sample the loop conformational ensemble before docking.

### ProteinMPNN — Why Autoregressive Decoding?

ProteinMPNN decodes amino acids one at a time in a random order (not N→C).
This matters because the conditional probability P(aa_i | backbone, context)
is much sharper when some neighbours are already assigned — the network
uses partially-assigned sequences to reduce uncertainty at remaining positions.

The **temperature parameter** controls entropy of the output distribution:
- T=0.1: nearly deterministic — samples near the maximum likelihood sequence
- T=0.5: more diverse — explores the sequence space around the target fold
- T=1.0: samples proportionally to the raw learned distribution

For experimental campaigns, T=0.1–0.2 works best for stability; T=0.3–0.5
for diversity screens exploring functionally distinct variants.

### Self-Consistency as a Filter

The scTM > 0.8 threshold came from Baker lab benchmarks showing that
experimentally successful designs almost always had scTM > 0.8 when tested
with AlphaFold2. This threshold has become the field standard for in-silico
pre-filtering before ordering synthetic genes.

### AlphaFold in Toxicology — Limitations

1. **No ligand in context**: AF2 predicts apo (unliganded) structures. Binding
sites may change conformation upon ligand binding (induced fit).
2. **No dynamics**: AF2 gives one static structure — not the ensemble of
conformations proteins explore. Allosteric effects may be missed.
3. **Homomers only** by default (AF2 multimer handles complexes).
4. **Training data cutoff**: AF2 may not predict novel folds with no homologs.
5. **pLDDT ≠ B-factor**: pLDDT measures model confidence, not thermal motion.


---
## ✅ Key Takeaways — AlphaFold2 & ProteinMPNN

1. **AlphaFold2 solved the 50-year protein folding problem** — pLDDT > 90 means the prediction is as accurate as a crystal structure for practical purposes. Low pLDDT often means the region is genuinely disordered, not a model failure.

2. **ESMFold is 60× faster with slightly lower accuracy** — use it for screening thousands of sequences. Use ColabFold/AlphaFold2 for publication-quality structures.

3. **ProteinMPNN samples sequences that fold to a given backbone** — the temperature parameter controls diversity vs fidelity. Active-site residues should be fixed. Target sequence recovery of 30-50%.

4. **Self-consistency (scTM > 0.8) is the in-silico validation standard** — fold the designed sequence, compare to the design backbone. This filter removes ~70% of failures before any wet-lab experiment.

5. **The pAE matrix is as important as pLDDT** — low pAE between two domains means AlphaFold is confident about their relative orientation. High inter-domain pAE = predicted disordered linker.

6. **For toxicology**: AlphaFold enables structure-based understanding of any protein target. Variant effect prediction, binding site prediction, and enzyme engineering are all now tractable for any target with a known sequence.

---

### Practical Next Steps
```bash
# 1. Run ESMFold on your protein of interest:
curl -X POST https://api.esmatlas.com/foldSequence/v1/pdb/ \
     -d "MKTLLLTLVVVTIVCLDLGYTMHSEAVFMKQIEVQLVESGGGDVQPGRSLR..."

# 2. Install ProteinMPNN:
git clone https://github.com/dauparas/ProteinMPNN
conda create -n proteinmpnn python=3.9
conda activate proteinmpnn
pip install torch

# 3. ColabFold (easiest AlphaFold2):
pip install colabfold[alphafold]
colabfold_batch sequences.fasta output_dir/

# 4. RFdiffusion (de novo design):
git clone https://github.com/RosettaCommons/RFdiffusion
```

---
*Part of the Python Ecosystem Tutorial Series | [hgoelgithub.github.io](https://hgoelgithub.github.io)*  
*Tools: AlphaFold2 (DeepMind) · ESMFold (Meta/FAIR) · ProteinMPNN (Baker Lab/UW) · RoseTTAFold (IPD/UW)*
